
# Taller — Titanic (Regresión Logística) con Pipeline + ColumnTransformer (ESQUELETO)

En esta tarea vas a **completar** un flujo profesional de clasificación para predecir `Survived`.

Tendrás tres archivos (en `data/`):
- `titanic_train_70.csv` (con etiquetas)
- `titanic_public_test_with_labels_15.csv` (con etiquetas, para experimentar)
- `titanic_private_test_without_labels_15.csv` (sin etiquetas, para la entrega)

🎯 Objetivo:
- Entrenar y evaluar tu modelo usando **train** y **public test**
- Generar `submission.csv` para el **private test**

✅ Reglas:
- Todo el preprocesamiento debe ir en un **Pipeline**
- Debes usar **Regresión Logística** como modelo final
- No inventes variables mirando el target en el private test (no existe)


## 1. Importar librerías

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
cd "/content/drive/MyDrive/Courses/AI_IS/5_linlog_reg"

/content/drive/MyDrive/Courses/AI_IS/5_linlog_reg


In [3]:

import os
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score
)

import matplotlib.pyplot as plt


## 2. Cargar datasets

In [4]:

DATA_DIR = "data"

train_path = os.path.join(DATA_DIR, "titanic_train_70.csv")
public_path = os.path.join(DATA_DIR, "titanic_public_test_with_labels_15.csv")
private_path = os.path.join(DATA_DIR, "titanic_private_test_without_labels_15.csv")

for p in [train_path, public_path, private_path]:
    assert os.path.exists(p), f"No encuentro el archivo: {p}"

train_df = pd.read_csv(train_path)
public_df = pd.read_csv(public_path)
private_df = pd.read_csv(private_path)

train_df.shape, public_df.shape, private_df.shape


((623, 12), (134, 12), (134, 11))

In [5]:

train_df.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,749,0,1,"Marvin, Mr. Daniel Warner",male,19.0,1,0,113773,53.1000,D30,S
1,46,0,3,"Rogers, Mr. William John",male,NaN,0,0,S.C./A.4. 23567,8.0500,NaN,S
2,29,1,3,"O'Dwyer, Miss. Ellen ""Nellie""",female,NaN,0,0,330959,7.8792,NaN,Q
3,634,0,1,"Parr, Mr. William Henry Marsh",male,NaN,0,0,112052,0.0000,NaN,S
4,404,0,3,"Hakkarainen, Mr. Pekka Pietari",male,28.0,1,0,STON/O2. 3101279,15.8500,NaN,S



## 3. Entender las variables (¡antes de modelar!)

Titanic trae columnas típicas como:

- **PassengerId**: identificador (NO explica supervivencia por sí mismo). Suele usarse solo para el submission.
- **Name**: texto con nombre (puede contener títulos como *Mr, Mrs, Miss*).
- **Sex**: sexo (categórica).
- **Age**: edad (numérica, con faltantes).
- **SibSp / Parch**: familiares a bordo (numéricas).
- **Ticket**: identificador de ticket (alta cardinalidad, puede introducir ruido).
- **Fare**: tarifa (numérica).
- **Cabin**: cabina (muchos faltantes, texto).
- **Embarked**: puerto de embarque (categórica).

👉 Tu trabajo: **decidir qué variables ayudan** y cuáles estorban.

### Pistas (no son reglas absolutas)
- Identificadores puros suelen aportar poco: `PassengerId`, `Ticket`.
- Texto libre puede ser difícil: `Name` (a menos que extraigas una señal simple como el título).
- Muchísimos faltantes: `Cabin` (a veces se descarta o se transforma a “tiene cabina: sí/no”).


### 3.1 Revisión rápida de tipos y faltantes

In [6]:

# Tipos de datos
train_df.dtypes


,0
PassengerId,int64
Survived,int64
Pclass,int64
Name,object
Sex,object
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64


In [7]:

# Porcentaje de valores faltantes por columna (train)
missing_pct = (train_df.isna().mean().sort_values(ascending=False) * 100).round(2)
missing_pct


,0
Cabin,78.17
Age,19.10
Embarked,0.32
PassengerId,0.00
Name,0.00
Pclass,0.00
Survived,0.00
Sex,0.00
Parch,0.00
SibSp,0.00



### 3.2 Distribución del target

En `train` y `public` existe `Survived`.  
Revisa si hay desbalance (no siempre es grave, pero importa para métricas).


In [8]:

TARGET = "Survived"
train_df[TARGET].value_counts(normalize=True).rename("train")


,train
Survived,
0,0.616372
1,0.383628


In [ ]:

public_df[TARGET].value_counts(normalize=True).rename("public")



## 4. Selección de variables (TODO)

Aquí vas a crear una lista de columnas para **eliminar** o **conservar**.

✅ Lo mínimo recomendado:
- Mantener variables claras: `Sex`, `Age`, `Pclass`, `Fare`, `SibSp`, `Parch`, `Embarked`
- No usar `PassengerId` como predictor (pero lo necesitas para el submission)
- Decidir qué hacer con `Name`, `Ticket`, `Cabin`

### TODO
1) Define una lista `DROP_COLS` con columnas que NO usarás como features.  
2) Crea `X_train`, `y_train`, `X_public`, `y_public`, `X_private` aplicando ese filtro.

💡 Consejo: empieza simple (descarta `Name`, `Ticket`, `Cabin`) y luego experimenta.


In [ ]:

# TODO (estudiante):
# DROP_COLS = [...]

# Ejemplo (solo como idea, cámbialo):
# DROP_COLS = ["PassengerId", "Name", "Ticket", "Cabin"]

DROP_COLS = None  # <- reemplaza

# TODO (estudiante): crea X/y para train y public, y X para private.
# Requisitos:
# - y_train = train_df[TARGET]
# - y_public = public_df[TARGET]
# - X_private NO tiene TARGET

# X_train = ...
# y_train = ...
# X_public = ...
# y_public = ...
# X_private = ...




## 5. Preprocesamiento con ColumnTransformer (TODO)

Usa:
- Numéricas: imputación + (opcional) escalado
- Categóricas: imputación + OneHotEncoder

### TODO
1) Identifica `num_cols` y `cat_cols` a partir de `X_train`  
2) Construye:
- `numeric_preprocess`
- `categorical_preprocess`
- `preprocess = ColumnTransformer(...)`


In [ ]:

# TODO (estudiante):
# num_cols = ...
# cat_cols = ...

# numeric_preprocess = Pipeline(...)
# categorical_preprocess = Pipeline(...)

# preprocess = ColumnTransformer(...)




## 6. Pipeline + Regresión Logística (TODO)

Crea un `Pipeline` con:
1) `("preprocess", preprocess)`
2) `("model", LogisticRegression(...))`

### TODO
- Crea `pipe`
- Entrena con `pipe.fit(X_train, y_train)`


In [ ]:

# TODO (estudiante):
# model = LogisticRegression(...)
# pipe = Pipeline(steps=[("preprocess", preprocess), ("model", model)])

# pipe.fit(X_train, y_train)



## 7. Evaluación en el Public Test (TODO)

Calcula:
- `classification_report`
- matriz de confusión
- (opcional) ROC-AUC si usas `predict_proba`

### TODO
- Genera `y_pred_public`
- Muestra métricas


In [ ]:

# TODO (estudiante):
# y_pred_public = pipe.predict(X_public)
# print(classification_report(y_public, y_pred_public, digits=4))

# cm = confusion_matrix(y_public, y_pred_public)
# ConfusionMatrixDisplay(cm).plot()
# plt.show()

# if hasattr(pipe, "predict_proba"):
#     y_proba_public = pipe.predict_proba(X_public)[:, 1]
#     print("ROC-AUC:", roc_auc_score(y_public, y_proba_public))



## 8. Entrenamiento final y submission (TODO)

Cuando decidas tu “mejor versión” del pipeline:

1) (Opcional) Entrena con **train + public** para tener más datos etiquetados.  
2) Predice el **private test sin etiquetas**.  
3) Genera `data/submission.csv` con:
- `PassengerId`
- `Survived` (predicción)

### TODO
- Entrenar final
- Predecir
- Guardar submission


In [ ]:

# TODO (estudiante):
# full_train_df = pd.concat([train_df, public_df], axis=0, ignore_index=True)
# X_full = full_train_df.drop(columns=[TARGET]).drop(columns=DROP_COLS)
# y_full = full_train_df[TARGET]
# pipe.fit(X_full, y_full)

# private_pred = pipe.predict(X_private)

# ID_COL = "PassengerId"
# assert ID_COL in private_df.columns, "PassengerId debe existir en el private test"
# submission = pd.DataFrame({
#     ID_COL: private_df[ID_COL].values,
#     TARGET: private_pred
# })
# out_path = os.path.join(DATA_DIR, "submission.csv")
# submission.to_csv(out_path, index=False)
# out_path



## 9. Ejercicios (elige al menos 3) — explica con evidencia

Usa métricas del **public test** para justificar tus respuestas (no solo opinión).

1) **Variables**: prueba incluir/excluir `Cabin`, `Name`, `Ticket`. ¿Qué cambia? ¿Por qué crees?
2) **Cabin simple**: crea una variable `HasCabin = Cabin.notna().astype(int)` y elimina `Cabin` original. ¿Mejora?
3) **Título del nombre**: extrae el título (Mr/Mrs/Miss/… ) desde `Name` y úsalo como categórica. ¿Mejora?
4) **Regularización**: prueba `C = 0.1, 1, 10`. ¿Qué pasa con precision/recall?
5) **Balanceo**: prueba `class_weight="balanced"`. ¿Mejora recall? ¿Pierde precisión?
6) **Escalado**: elimina `StandardScaler`. ¿Qué ocurre en regresión logística?

Plantilla para cada ejercicio:
- Cambio:
- Métricas (public):
- Explicación (2–5 líneas):
